In [ ]:
# ------------------------------------
# Import modules
# ------------------------------------

# Core
import numpy as np
import pandas as pd

# XGBoost
from xgboost import XGBRegressor

# Scikit-learn: model selection
from sklearn.model_selection import RandomizedSearchCV, RepeatedKFold

# Scikit-learn: preprocessing & pipelines
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Scikit-learn: models
from sklearn.linear_model import MultiTaskLasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor


In [3]:
# -----------------------------
# Import local functions
# -----------------------------
import sys
import os

# Get the absolute path to your src folder
module_path = os.path.abspath(os.path.join('..', 'src'))

if module_path not in sys.path:
    sys.path.append(module_path)

from cleaning_and_helpers import plot_test_preds, split_project, multioutput_rmse, evaluate_model


In [ ]:
# ------------------------------------
# Set seed
# ------------------------------------
np.random.seed(1298)
seed = 1298

In [ ]:
# ------------------------------------
# Import pre-processed train/test data
# ------------------------------------

# The three projects were split to retain 20% for testing. The below code is an example of the 
# preprocessing, but because the latitude and longitude coordinates for the PNW Forests project are
# private, we are only providing access to the data for the other two projects. All analyses can be 
# run using just the other two datasets. Model outputs below reflect training on the full dataset.

# Train test split for each project
# seed = 1298
# test_size = 0.20

# Function to split each project
# def split_project(X, y, test_size, random_state):
#     return train_test_split(X, y, test_size=test_size, random_state=random_state)

# Split each project
# SI_X_train, SI_X_test, SI_y_train, SI_y_test = split_project(SI_X, SI_y, test_size, seed)
# PNW_X_train, PNW_X_test, PNW_y_train, PNW_y_test = split_project(PNW_X, PNW_y, test_size, seed)
# CA_X_train, CA_X_test, CA_y_train, CA_y_test = split_project(CA_X, CA_y, test_size, seed)

# Concatenate train and test splits from all projects
# X_train = np.concatenate([SI_X_train, PNW_X_train, CA_X_train], axis=0)
# y_train = np.concatenate([SI_y_train, PNW_y_train, CA_y_train], axis=0)

# X_test = np.concatenate([SI_X_test, PNW_X_test, CA_X_test], axis=0)
# y_test = np.concatenate([SI_y_test, PNW_y_test, CA_y_test], axis=0)

# Standardize X and y separately for train/test to prevent leakage
# sc_X = StandardScaler()
# sc_y = StandardScaler()

# X_train = sc_X.fit_transform(X_train)
# y_train = sc_y.fit_transform(y_train)

# X_test = sc_X.transform(X_test)
# y_test = sc_y.transform(y_test)

# Import data
X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')

X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

In [ ]:
# -----------------------------
# Custom RMSE scorer
# -----------------------------

rmse_scorer = make_scorer(multioutput_rmse, greater_is_better=False)

# -----------------------------
# Cross-validation setup
# -----------------------------
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=1298)

# -----------------------------
# Models and param grids
# -----------------------------
models = {
    "MultiTaskLasso": (
        linear_model.MultiTaskLasso(max_iter=10000),
        {
            'estimator__alpha': np.arange(0.00, 1.0, 0.01)
        }
    ),

    "SVR": (
        MultiOutputRegressor(SVR()),
        {
            'estimator__estimator__C': uniform(0.1, 10),  # Uniform distribution between 0.1 and 10
            'estimator__estimator__kernel': ['linear', 'rbf', 'poly'],
            'estimator__estimator__gamma': ['scale', 'auto']
        }  
    ),

    "KNN": (
        MultiOutputRegressor(KNeighborsRegressor()),
        {
            'estimator__estimator__n_neighbors': np.arange(2, 50, 1),  # Number of neighbors (2 to 50)
            'estimator__estimator__weights': ['uniform', 'distance'],  # Weighting method
            'estimator__estimator__p': [1, 2],  # Manhattan or Euclidean distance
            'estimator__estimator__algorithm': ['auto']  # Default to 'auto'
        }
    ),

    "DecisionTree": (
        MultiOutputRegressor(DecisionTreeRegressor(random_state=seed)),
        {
            'estimator__estimator__max_depth': [None] + list(range(3, 21, 1)),      # Depths from 3 to 20 + unlimited depth
            'estimator__estimator__min_samples_split': [2, 5, 10, 20, 50],          # Minimum samples to split
            'estimator__estimator__min_samples_leaf': [1, 2, 5, 10, 20, 50],        # Minimum samples in a leaf
            'estimator__estimator__max_features': ['auto', 'sqrt', 'log2', None],   # Features considered at each split
            'estimator__estimator__max_leaf_nodes': [None] + list(range(10, 101, 10))  # Maximum number of leaf nodes

        }
    ),

    "RandomForest": (
        MultiOutputRegressor(RandomForestRegressor(random_state=seed)),
        {
            'estimator__estimator__n_estimators': [50, 100, 200, 500, 1000],           # Number of trees
            'estimator__estimator__max_depth': [None] + list(range(10, 51, 10)),   # Maximum depth of trees
            'estimator__estimator__min_samples_split': [2, 5, 10, 20, 50],             # Minimum samples to split a node
            'estimator__estimator__min_samples_leaf': [1, 2, 5, 10, 20, 50],          # Minimum samples in a leaf node
            'estimator__estimator__max_features': ['auto', 'sqrt', 'log2', None],  # Features considered for splits
          
        }  
    ),

    "XGBoost": (
        MultiOutputRegressor(XGBRegressor(objective='reg:squarederror', random_state=seed)),
        {
            'estimator__estimator__n_estimators': [100, 200, 500, 1000],          # Number of trees
            'estimator__estimator__learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],  # Learning rate
            'estimator__estimator__max_depth': [3, 5, 7, 10, 15, 20],            # Maximum tree depth
            'estimator__estimator__min_child_weight': [1, 5, 10, 20],            # Minimum child weight
            'estimator__estimator__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],        # Subsampling ratio
            'estimator__estimator__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0], # Features per tree
            'estimator__estimator__gamma': [0, 0.1, 0.2, 0.5, 1, 5],             # Minimum loss reduction
            'estimator__estimator__reg_alpha': [0, 0.01, 0.1, 1, 10, 100],       # L1 regularization
            'estimator__estimator__reg_lambda': [1, 10, 50, 100]                 # L2 regularization

        }  
    )
}



# -----------------------------
# Run GridSearchCV for all models
# -----------------------------
results = {}

for name, (model, param_grid) in models.items():
    print(f"Training model: {name}")
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('estimator', model)
    ])
    
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid,
        scoring=rmse_scorer,
        cv=cv,
        n_iter=50,    
        n_jobs=-1,
        verbose=1,
        random_state=1298
    )
    
    search.fit(X_train, y_train)
    results[name] = search

#--------------------------
# Compare Results
#--------------------------
for name, search in results.items():
    print(f"{name}: Best CV RMSE = {-search.best_score_:.3f}")

Training model: MultiTaskLasso
Fitting 15 folds for each of 50 candidates, totalling 750 fits
Training model: SVR
Fitting 15 folds for each of 50 candidates, totalling 750 fits
Training model: KNN
Fitting 15 folds for each of 50 candidates, totalling 750 fits
Training model: DecisionTree
Fitting 15 folds for each of 50 candidates, totalling 750 fits
Training model: RandomForest
Fitting 15 folds for each of 50 candidates, totalling 750 fits
Training model: XGBoost
Fitting 15 folds for each of 50 candidates, totalling 750 fits
MultiTaskLasso: Best CV RMSE = 0.804
SVR: Best CV RMSE = 0.714
KNN: Best CV RMSE = 0.649
DecisionTree: Best CV RMSE = 0.488
RandomForest: Best CV RMSE = 0.340
XGBoost: Best CV RMSE = 0.435


In [ ]:
# ------------------------------------
# Print best parameters for each model
# ------------------------------------
for name, search in results.items():
    print(f"{name} best params:")
    print(search.best_params_)
    print()

MultiTaskLasso best params:
{'estimator__alpha': 0.08}

SVR best params:
{'estimator__estimator__C': 2.847162213648956, 'estimator__estimator__gamma': 'auto', 'estimator__estimator__kernel': 'rbf'}

KNN best params:
{'estimator__estimator__weights': 'uniform', 'estimator__estimator__p': 2, 'estimator__estimator__n_neighbors': 3, 'estimator__estimator__algorithm': 'auto'}

DecisionTree best params:
{'estimator__estimator__min_samples_split': 50, 'estimator__estimator__min_samples_leaf': 1, 'estimator__estimator__max_leaf_nodes': None, 'estimator__estimator__max_features': 'sqrt', 'estimator__estimator__max_depth': None}

RandomForest best params:
{'estimator__estimator__n_estimators': 200, 'estimator__estimator__min_samples_split': 5, 'estimator__estimator__min_samples_leaf': 1, 'estimator__estimator__max_features': 'log2', 'estimator__estimator__max_depth': None}

XGBoost best params:
{'estimator__estimator__subsample': 0.8, 'estimator__estimator__reg_lambda': 1, 'estimator__estimator_

In [ ]:

# Define model mapping (must match what was passed into GridSearchCV)
model_classes = {
    "MultiTaskLasso": MultiTaskLasso,
    "SVR": lambda **kwargs: MultiOutputRegressor(SVR(**kwargs)),
    "KNN": lambda **kwargs: MultiOutputRegressor(KNeighborsRegressor(**kwargs)),
    "DecisionTree": lambda **kwargs: MultiOutputRegressor(DecisionTreeRegressor( **kwargs)),
    "RandomForest": lambda **kwargs: MultiOutputRegressor(RandomForestRegressor( **kwargs)),
    "XGBoost": lambda **kwargs: MultiOutputRegressor(XGBRegressor(objective="reg:squarederror", **kwargs))
}

# Collect results
all_metrics = []

for name, search in results.items():
    best_params = search.best_params_
    
    # Flatten parameter dict for instantiation (remove 'estimator__estimator__' prefix)
    flat_params = {
        k.replace("estimator__estimator__", "").replace("estimator__", ""): v
        for k, v in best_params.items()
    }

    metrics = evaluate_model(
        name=name,
        model_class=model_classes[name],
        best_params=flat_params,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test
    )
    all_metrics.append(metrics)


Evaluating MultiTaskLasso...
Evaluating SVR...
Evaluating KNN...
Evaluating DecisionTree...
Evaluating RandomForest...
Evaluating XGBoost...


In [ ]:
# ------------------------------------
# Save output as csv
# ------------------------------------

results_df = pd.DataFrame([{
    "Model": m["model"],
    "R²": m["r2"],
    "RMSE": m["rmse"],
    "MAE": m["mae"],
    "Avg Geodesic Error (km)": m["avg_km_error"],
    "SE Geodesic Error (km)": m["se_km_error"]
} for m in all_metrics])

results_df.to_csv("../tables/raw/raw_best_case_tuned_results.csv", index=False)

results_df



,Model,R²,RMSE,MAE,Avg Geodesic Error (km),SE Geodesic Error (km)
0,MultiTaskLasso,0.448394,0.745991,0.228303,76.467168,89.006972
1,SVR,0.611294,0.630388,0.095733,59.156881,79.260407
2,KNN,0.709883,0.538869,0.015321,36.048914,76.934562
3,DecisionTree,0.810490,0.440076,0.019726,25.324499,64.851450
4,RandomForest,0.884195,0.344840,0.031582,24.924802,48.151837
5,XGBoost,0.858185,0.381605,0.072625,32.150896,50.605773
